In [2]:
import osmium
from pvlib import solarposition
from shapely.geometry import LineString, Polygon, Point
from pyproj import CRS, Transformer
from shapely.ops import transform, unary_union
from shapely.affinity import translate
import numpy as np
from tqdm import tqdm
import cProfile
import pstats
import pandas as pd

In [3]:
if __name__ == "__main__":
    profiler = cProfile.Profile()
    profiler.enable()
    # Définir les systèmes de coordonnées
    crs_latlon = CRS("EPSG:4326")  # WGS84 (latitude/longitude)
    crs_projected = CRS("EPSG:32630")  # UTM pour projection en mètres
    transformer_to_meters = Transformer.from_crs(crs_latlon, crs_projected, always_xy=True)


    class WayModifier(osmium.SimpleHandler):
        def __init__(self, input_file, output_pbf):
            super().__init__()
            self.time = pd.DatetimeIndex([pd.Timestamp.now().tz_localize("Europe/Paris")])
            self.transformer_to_meters = transformer_to_meters
            self.road_width = 10  # Largeur moyenne des routes
            self.input_file = input_file
            self.pbf_writer = osmium.SimpleWriter(output_pbf)
            self.modified = False
            self.buildings, self.trees, self.center = self.get_data(input_file)  # Récupération des bâtiments et des arbres une seule fois
            # Building parameters
            self.building_area_spread = 50  # Rayon pour récupérer les bâtiments autour des routes
            self.default_building_height = 4  # Hauteur par défaut des bâtiments (mètres)
            self.level_height = 2.8  # Hauteur moyenne par étage (mètres)
            # Tree parameters
            self.tree_area_spread = 20  # Rayon pour récupérer les arbres autour des routes
            self.default_tree_height = 5  # Hauteur par défaut des arbres (mètres)
            self.default_tree_width = 3  # Largeur par défaut des arbres (mètres)
            self.x_default_tree = 0
            self.y_default_tree = 0
            self.default_shadow_tree = self.create_default_shadow_tree()

        def get_data(self, input_file):
            '''Get all buildings and trees from the input file, 
            and approximate center of the PBF'''
            # Étape 1 : Compter le nombre total de bâtiments pour la progression
            class CounterHandler(osmium.SimpleHandler):
                def __init__(self):
                    super().__init__()
                    self.count = 0

                def way(self, w):
                    if "building" in w.tags:
                        self.count += 1

            counter = CounterHandler()
            counter.apply_file(input_file, locations=True)

            # Étape 2 : Handler avec barre de progression
            class DataHandler(osmium.SimpleHandler):
                def __init__(self, total_ways, sample_rate=1000):
                    super().__init__()
                    self.buildings = []
                    self.trees = []
                    self.pbar = tqdm(total=total_ways, desc="Processing Buildings and trees", unit="way")
                    
                    # To get approx center
                    self.min_lon, self.min_lat = float('inf'), float('inf')
                    self.max_lon, self.max_lat = float('-inf'), float('-inf')
                    self.sample_rate = sample_rate
                    self.node_count = 0

                def way(self, w):
                    if "building" in w.tags:
                        coords = [(n.lon, n.lat) for n in w.nodes]
                        self.buildings.append(transform(transformer_to_meters.transform, Polygon(coords)))
                    self.pbar.update(1)  # Mise à jour de la barre de progression
                
                def node(self, n):
                    if "natural" in n.tags and n.tags["natural"] == "tree":
                        if self.pbar is None:  # Initialise tqdm seulement au premier arbre
                            self.pbar = tqdm(desc="Processing trees", unit="tree")
                        lon, lat = n.lon, n.lat
                        point = transform(transformer_to_meters.transform, Point(lon, lat))
                        self.trees.append(point)

                    # To get approx center
                    if self.node_count % self.sample_rate == 0:  # Prend seulement 1 nœud sur sample_rate
                        self.min_lon = min(self.min_lon, n.lon)
                        self.min_lat = min(self.min_lat, n.lat)
                        self.max_lon = max(self.max_lon, n.lon)
                        self.max_lat = max(self.max_lat, n.lat)
                    self.node_count += 1
                
                def close(self):
                    self.pbar.close()  # Ferme la barre de progression

            handler = DataHandler(counter.count)
            handler.apply_file(input_file, locations=True)

            # To get approx center
            if handler.min_lon == float('inf'):
                print("Impossible de déterminer le centre : aucun point trouvé.")
                return None
            center_lon = (handler.min_lon + handler.max_lon) / 2
            center_lat = (handler.min_lat + handler.max_lat) / 2
            center = Point(center_lon, center_lat)

            handler.close()  # Fermer proprement la barre de progression

            return handler.buildings, handler.trees, center

        def create_default_shadow_tree(self):
            """Create a tree approximately in the center of the map and project its shadow"""
            half_width_tree = self.default_tree_width/2
            center = self.center
            center_meters = transform(transformer_to_meters.transform, center)
            self.x_default_tree, self.y_default_tree = center_meters.x, center_meters.y
            tree_base = Polygon([
                    (self.x_default_tree - half_width_tree, self.y_default_tree - half_width_tree),
                    (self.x_default_tree + half_width_tree, self.y_default_tree - half_width_tree),
                    (self.x_default_tree + half_width_tree, self.y_default_tree + half_width_tree),
                    (self.x_default_tree - half_width_tree, self.y_default_tree + half_width_tree)
                ])
            solar_position = solarposition.get_solarposition(self.time, center.y, center.x)
            sun_azimuth = solar_position["azimuth"].values[0]
            sun_elevation = solar_position["elevation"].values[0]
            return self.project_shadow_tree(tree_base, self.default_tree_height, sun_elevation, sun_azimuth)

        def way(self, w):
            if "highway" in w.tags:
                # in the graphhopper code, shade percentage should be an integer that's why we have to round the value 
                shade_value = round(self.calculate_shade(w))
                #print(shade_value)
                tags = list(w.tags)  # Copier les tags existants
                tags.append(osmium.osm.Tag("shade:percentage", f"{shade_value}%"))  # Ajouter le nouveau tag

                if shade_value >= 75 and shade_value <= 100:
                    tags.append(osmium.osm.Tag("shade", "yes"))
                elif shade_value < 75 and shade_value >= 15 :
                    tags.append(osmium.osm.Tag("shade", "partial"))
                elif shade_value>= 0 and shade_value < 15 :
                    tags.append(osmium.osm.Tag("shade", "no"))

                new_way = osmium.osm.mutable.Way(w)
                new_way.tags = tags

                self.pbf_writer.add_way(new_way)
                self.modified = True
            else:
                self.pbf_writer.add_way(w)

        def node(self, n):
            self.pbf_writer.add_node(n)

        def relation(self, r):
            self.pbf_writer.add_relation(r)

        def close(self):
            self.pbf_writer.close()

        def get_closest_points_to_road(self, building_base, road):
            base_coords = list(building_base.exterior.coords)
            distances = [(Point(coord).distance(road), coord) for coord in base_coords]
            distances.sort(key=lambda x: x[0])
            return [distances[0][1], distances[1][1]]

        def project_shadow(self, building, building_height, sun_elevation, sun_azimuth, road):
            if sun_elevation > 0:
                shadow_length = building_height / np.tan(np.radians(sun_elevation))
            else:
                return Polygon([])  # Pas d'ombre si le soleil est sous l'horizon
            azimuth_radians = np.radians(sun_azimuth)
            closest_points = self.get_closest_points_to_road(building, road)
            projected_points = [
                (x + shadow_length * np.cos(azimuth_radians), y + shadow_length * np.sin(azimuth_radians))
                for x, y in closest_points
            ]

            return Polygon([
                closest_points[0],
                projected_points[0],
                projected_points[1],
                closest_points[1]
            ])
        
        def project_shadow_tree(self, tree_base, tree_height, sun_elevation, sun_azimuth):
            if sun_elevation > 0:
                shadow_length = tree_height / np.tan(np.radians(sun_elevation))
            else:
                return Polygon([])  # Pas d'ombre si le soleil est sous l'horizon

            azimuth_radians = np.radians(sun_azimuth)
            
            base_coords = list(tree_base.exterior.coords)[:4]  # Prendre les 4 premiers points de l'arbre
            shadow_parts = []
            for i in range(len(base_coords)): # Générer les ombres pour chaque côté du carré
                p1 = base_coords[i]
                p2 = base_coords[(i + 1) % len(base_coords)] 
                # Projeter ces deux points
                p1_proj = (p1[0] + shadow_length * np.cos(azimuth_radians), p1[1] + shadow_length * np.sin(azimuth_radians))
                p2_proj = (p2[0] + shadow_length * np.cos(azimuth_radians), p2[1] + shadow_length * np.sin(azimuth_radians))
                quad = Polygon([p1, p2, p2_proj, p1_proj])
                shadow_parts.append(quad)
            shadow_polygon = unary_union(shadow_parts)

            return shadow_polygon

        def calculate_shade(self, way):
            # Convertir la route en polygone (zone impactée par l'ombre)
            print("Calcul des ombres des bâtiments...")
            coords = [(n.lon, n.lat) for n in way.nodes]
            way_line_latlon = LineString(coords)
            way_line_meters = transform(self.transformer_to_meters.transform, way_line_latlon)
            road_area = way_line_meters.buffer(self.road_width)

            # Obtenir la position du soleil
            #longitude, latitude = way_line_latlon.centroid.x, way_line_latlon.centroid.y
            solar_position = solarposition.get_solarposition(self.time, coords[0][1], coords[0][0])
            sun_azimuth = solar_position["azimuth"].values[0]
            sun_elevation = solar_position["elevation"].values[0]

            all_shadows = [] # Stocker tous les polygones d'ombre
            
            # Calculer l'ombre projetée par les bâtiments environnants
            
            for building in tqdm(self.buildings, desc="Bâtiments", unit="bâtiment"):
                if way_line_meters.distance(building.centroid) < self.building_area_spread:
                    # Estimer la hauteur du bâtiment
                    building_height = self.default_building_height
                    if "height" in way.tags:
                        building_height = float(way.tags["height"])
                    elif "building:levels" in way.tags:
                        building_height = self.level_height * float(way.tags["building:levels"])
                    shadow_polygon = self.project_shadow(building, building_height, sun_elevation, sun_azimuth, way_line_meters)
                    all_shadows.append(shadow_polygon)

            merged_shadows_1 = unary_union(all_shadows)
            intersection = road_area.intersection(merged_shadows_1)
            #print(f"🛑 Ombre après bâtiments: {intersection.area}")

            # Calculer l'ombre projetée par les arbres environnants
            print("Calcul des ombres des arbres...")
            for tree in tqdm(self.trees, desc="Arbres", unit="arbre"):
                if road_area.distance(tree) < self.tree_area_spread:
                    tree_shadow = translate(self.default_shadow_tree, xoff=tree.x-self.x_default_tree, yoff=tree.y-self.y_default_tree)
                    all_shadows.append(tree_shadow)

            merged_shadows = unary_union(all_shadows)
            intersection = road_area.intersection(merged_shadows)
            #print(f"🌳 Ombre après bâtiments et arbres: {intersection.area}")
            shadow_area = intersection.area

            return (shadow_area / road_area.area) * 100 if road_area.area > 0 else 0


    # Chemins des fichiers
    input_file = "C:\\Users\\yahia\\Documents\\IMT\\procom_calcul\\test.pbf"
    output_pbf = "C:\\Users\\yahia\\Documents\\IMT\\procom_calcul\\test_updated.pbf"

    modifier = WayModifier(input_file, output_pbf)
    modifier.apply_file(input_file, locations=True)
    modifier.close()

    if modifier.modified:
        print(f"Les routes ont été modifiées et ajoutées au fichier {output_pbf}.")
    else:
        print(f"Aucune route n'a été trouvée ou modifiée.")
    

    profiler.disable()
    stats = pstats.Stats(profiler)
    stats.strip_dirs().sort_stats("cumtime").print_stats(20)

Processing Buildings and trees: 3308way [00:00, 4525.24way/s]                         


Calcul des ombres des bâtiments...


Bâtiments: 100%|██████████| 1179/1179 [00:00<00:00, 11947.94bâtiment/s]


Calcul des ombres des arbres...


Arbres: 100%|██████████| 1003/1003 [00:00<00:00, 15401.10arbre/s]


...

Calcul des ombres des bâtiments...


Bâtiments: 100%|██████████| 1179/1179 [00:00<00:00, 29811.75bâtiment/s]


Calcul des ombres des arbres...


Arbres: 100%|██████████| 1003/1003 [00:00<00:00, 45040.65arbre/s]


Les routes ont été modifiées et ajoutées au fichier C:\Users\yahia\Documents\IMT\procom_calcul\test_updated.pbf.
         46819841 function calls (44867948 primitive calls) in 205.878 seconds

   Ordered by: cumulative time
   List reduced from 932 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  4025301   30.045    0.000  105.885    0.000 decorators.py:62(wrapped)
2473526/1916622    6.461    0.000   68.205    0.000 base.py:332(distance)
  2473527   41.896    0.000   41.899    0.000 measurement.py:47(distance)
  1245024    2.176    0.000   27.461    0.000 base.py:368(centroid)
    14902    1.162    0.000   17.712    0.001 1439489926.py:151(get_closest_points_to_road)
  1245024   15.002    0.000   15.002    0.000 constructive.py:258(centroid)
     1057    0.066    0.000   13.332    0.013 solarposition.py:281(spa_python)
2306303/1774587    4.369    0.000   12.072    0.000 std.py:1160(__iter__)
    13500    0.198    0.000   12.047   